# Walkthrough: hiểu trọng số, loss, gradient descent và LoRA bằng Python

Notebook này không tải Qwen, không cần GPU, và không train model thật. Mục tiêu là hiểu các ý tưởng nhỏ trước:

- Trọng số là gì.
- Loss giảm bằng cách nào.
- Token/embedding hoạt động ra sao ở mức trực giác.
- LoRA giữ trọng số gốc cố định và chỉ train adapter nhỏ như thế nào.

Sau notebook này, đọc lại `src/prepare_data.py`, `src/train_lora.py`, và `src/utils.py` sẽ dễ hơn nhiều.

In [ ]:
import math
import numpy as np

np.set_printoptions(precision=4, suppress=True)

## 1. Một model nhỏ nhất: `y = x * weight + bias`

Trong model thật có hàng tỷ trọng số. Ở đây ta bắt đầu với 2 trọng số: `weight` và `bias`.

Giả sử dữ liệu thật đi theo luật:

```text
y = 2 * x + 1
```

Model chưa biết luật này. Training sẽ chỉnh `weight` và `bias` để dự đoán gần đúng hơn.

In [ ]:
x = np.array([0.0, 1.0, 2.0, 3.0])
y_true = np.array([1.0, 3.0, 5.0, 7.0])

weight = 0.2
bias = 0.0

def predict(x, weight, bias):
    return x * weight + bias

def mse_loss(y_pred, y_true):
    return np.mean((y_pred - y_true) ** 2)

y_pred = predict(x, weight, bias)
print('prediction:', y_pred)
print('loss:', mse_loss(y_pred, y_true))

## 2. Gradient descent: cập nhật trọng số để loss giảm

Với mean squared error:

```text
loss = mean((prediction - y_true)^2)
prediction = x * weight + bias
```

Gradient cho biết nếu tăng/giảm trọng số thì loss đổi theo hướng nào. Công thức update:

```text
weight = weight - learning_rate * grad_weight
bias = bias - learning_rate * grad_bias
```

In [ ]:
weight = 0.2
bias = 0.0
learning_rate = 0.05

for step in range(30):
    y_pred = predict(x, weight, bias)
    error = y_pred - y_true
    loss = mse_loss(y_pred, y_true)

    grad_weight = np.mean(2 * error * x)
    grad_bias = np.mean(2 * error)

    weight -= learning_rate * grad_weight
    bias -= learning_rate * grad_bias

    if step % 5 == 0 or step == 29:
        print(f'step={step:02d} loss={loss:.4f} weight={weight:.4f} bias={bias:.4f}')

Trong `src/train_lora.py`, `SFTTrainer` làm phiên bản lớn hơn của vòng lặp này:

- Forward: model dự đoán token tiếp theo.
- Tính loss: câu trả lời đúng có xác suất cao chưa.
- Backward: tính gradient.
- Optimizer update: cập nhật trọng số trainable.

Khác biệt lớn: repo không update toàn bộ base model, mà update LoRA adapter.

## 3. Cross-entropy trực giác

Language model dự đoán token kế tiếp bằng phân phối xác suất. Nếu token đúng có xác suất `p`, loss token đó là:

```text
-log(p)
```

Xác suất đúng càng cao thì loss càng thấp.

In [ ]:
for p in [0.90, 0.50, 0.10, 0.01]:
    print(f'p(correct)={p:>4} -> cross_entropy={-math.log(p):.4f}')

Repo dùng causal language modeling. Với mỗi vị trí token trong completion, model học tăng xác suất token đúng. Trong `src/train_lora.py`, `completion_only_loss=True` giúp loss chỉ tính trên phần assistant cần học trả lời, không tính trên toàn bộ prompt.

## 4. Token và embedding bản nhỏ

Tokenizer thật của Qwen phức tạp hơn ví dụ này nhiều. Nhưng ý tưởng cơ bản là:

```text
text -> token ids -> embedding vectors
```

`src/prepare_data.py` dùng tokenizer của base model để biến dữ liệu thành đúng chat template trước khi train.

In [ ]:
vocab = {'xin': 0, 'chao': 1, 'model': 2, '<eos>': 3}
embedding_table = np.array([
    [0.10, 0.20, 0.30],
    [0.00, 0.40, 0.10],
    [0.50, 0.10, 0.20],
    [0.00, 0.00, 0.00],
])

tokens = ['xin', 'chao', '<eos>']
token_ids = [vocab[token] for token in tokens]
vectors = embedding_table[token_ids]

print('tokens:', tokens)
print('token_ids:', token_ids)
print('embedding vectors:\n', vectors)

Trong repo, `render_training_record()` ở `src/utils.py` tạo `prompt` bằng:

```python
tokenizer.apply_chat_template(..., add_generation_prompt=True)
```

Điều này giúp dữ liệu train giống format chat mà base model đã quen.

## 5. LoRA bằng ma trận nhỏ

Giả sử một lớp neural network có trọng số gốc `W_base`. Fine-tune toàn bộ nghĩa là update trực tiếp `W_base`.

LoRA làm khác:

```text
W_new = W_base + delta_W
delta_W = B @ A
```

`W_base` được giữ cố định. Ta chỉ train hai ma trận nhỏ `A` và `B`.

In [ ]:
rng = np.random.default_rng(42)

# Input có 3 chiều, output có 2 chiều.
x_batch = np.array([
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 0.0],
])

W_base = np.array([
    [0.30, -0.20, 0.10],
    [0.00,  0.20, 0.40],
])

# Target giả lập: ta muốn lớp này cho output hơi khác base model.
y_target = np.array([
    [0.70, 0.45],
    [0.05, 0.90],
    [0.30, 0.25],
])

rank = 1
A = rng.normal(scale=0.01, size=(rank, 3))
B = np.zeros((2, rank))
lr = 0.5

def lora_forward(x_batch, W_base, A, B):
    delta_W = B @ A
    W_new = W_base + delta_W
    return x_batch @ W_new.T

for step in range(80):
    y_pred = lora_forward(x_batch, W_base, A, B)
    error = y_pred - y_target
    loss = np.mean(error ** 2)

    # d_loss/d_W_new cho MSE và linear layer.
    grad_W = (2 / error.size) * error.T @ x_batch

    # delta_W = B @ A
    grad_B = grad_W @ A.T
    grad_A = B.T @ grad_W

    B -= lr * grad_B
    A -= lr * grad_A

    if step in [0, 1, 2, 5, 10, 20, 40, 79]:
        print(f'step={step:02d} loss={loss:.6f}')

print('\nW_base giu co dinh:\n', W_base)
print('\ndelta_W hoc duoc:\n', B @ A)
print('\nW_new = W_base + delta_W:\n', W_base + B @ A)

Điều vừa xảy ra giống trực giác LoRA trong repo:

- `W_base` không đổi.
- `A` và `B` thay đổi để tạo `delta_W`.
- Adapter nhỏ nhưng vẫn có thể đổi hành vi output.

Trong `src/train_lora.py`, repo đặt:

```python
lora_r = 16
lora_alpha = 32
lora_dropout = 0.05
target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
```

Notebook dùng `rank = 1` để dễ nhìn; repo dùng rank 16 để adapter có năng lực học tốt hơn.

## 6. Map notebook này sang repo thật

| Ý tưởng trong notebook | Trong repo |
| --- | --- |
| `x`, `y_true` | JSONL có `instruction`, `input`, `output` |
| Token ids / embedding | `load_tokenizer()` và `apply_chat_template()` trong `src/utils.py` |
| Loss | Cross-entropy trên completion token trong `SFTTrainer` |
| Gradient update | Optimizer trong `SFTTrainer` |
| Giữ `W_base` cố định | Base model Qwen giữ nguyên |
| Train `A`, `B` | Train LoRA adapter qua `peft.LoraConfig` |
| `delta_W = B @ A` | LoRA adapter được lưu ở `outputs/.../final_adapter` |

Các lệnh thật để nối từ học sang chạy:

```bash
python src/prepare_data.py
python src/train_lora.py --config configs/rtx4060ti_8gb_mail_agent.yaml --max_steps 50
python src/eval.py
python src/chat.py
```

## 7. Câu hỏi tự kiểm tra

1. Nếu `learning_rate` quá cao thì điều gì có thể xảy ra với loss?
2. Vì sao `gradient_accumulation_steps=16` hữu ích khi `per_device_train_batch_size=1`?
3. Vì sao đổi `base_model` thì nên chạy lại `prepare_data.py`?
4. Adapter LoRA khác gì merged full model?
5. Trong ví dụ LoRA, chuyện gì xảy ra nếu cả `A` và `B` đều khởi tạo bằng 0?